# 🏆 Pokemon Win Prediction Assignment
This notebook analyzes Pokemon stats and battle outcomes to predict win percentages using Machine Learning.

### 🛠️ Tasks:
1. Data Cleaning & Engineering (Win Percentage calculation)
2. Exploratory Data Analysis (EDA)
3. Machine Learning Modeling (Regression)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Set visual style
sns.set(style='whitegrid')
print("Libraries loaded.")

## 1. Data Preparation
We load the `pokemon.csv` and `combats.csv` files, fix missing values, and calculate win percentages.

In [ ]:
# Load datasets
try:
    pokemon = pd.read_csv('pokemon.csv')
    combats = pd.read_csv('combats.csv')
    print("Datasets loaded successfully.")
except FileNotFoundError:
    print("Error: pokemon.csv or combats.csv not found. Please ensure files are in the same directory.")

# 1. Fix missing Name for Pokemon #62
pokemon.loc[pokemon['#'] == 62, 'Name'] = "Primeape"

# 2. Handle NaN in Type 2
pokemon['Type 2'] = pokemon['Type 2'].fillna("None")

pokemon.head()

### Feature Engineering: Calculate Win Percentage

In [ ]:
# Calculate total battles for each pokemon
total_fights_1 = combats['First_pokemon'].value_counts()
total_fights_2 = combats['Second_pokemon'].value_counts()
total_battles = total_fights_1.add(total_fights_2, fill_value=0)

# Calculate total wins for each pokemon
wins = combats['Winner'].value_counts()

# Calculate win percentage
win_percentage = (wins / total_battles) * 100

# Create a dataframe for win stats
win_stats = pd.DataFrame({
    '#': win_percentage.index,
    'Win %': win_percentage.values
})

# Merge with original pokemon data
df = pd.merge(pokemon, win_stats, on='#', how='left')

# Fill win % for pokemon with zero battles or wins
df['Win %'] = df['Win %'].fillna(0)

print("Win percentages calculated and merged.")
df[['Name', 'HP', 'Attack', 'Speed', 'Win %']].head()

## 2. Exploratory Analysis & Visualization

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 8))
stats_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Win %']
sns.heatmap(df[stats_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Between Stats and Win Percentage")
plt.show()

print("Finding: Speed and Attack show the strongest positive correlation with Win Percentage.")

In [ ]:
# Pairplot for stats vs Win %
sns.pairplot(df, x_vars=['HP', 'Attack', 'Speed', 'Defense'], y_vars=['Win %'], height=4)
plt.suptitle("Pokemon Stats vs Win Percentage", y=1.05)
plt.show()

In [ ]:
# Top 10 Pokemon by Win Percentage
top_10 = df.sort_values(by='Win %', ascending=False).head(10)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_10, x='Win %', y='Name', palette='viridis')
plt.title("Top 10 Strongest Pokemon by Win Rate")
plt.show()

print("Top 10 Pokemon stats preview:")
top_10[['Name', 'Type 1', 'Speed', 'Win %']]

## 3. Machine Learning: Predicting Win Percentage

In [ ]:
# Prepare data
features = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation', 'Legendary']
X = df[features]
y = df['Win %']

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

# Train and evaluate
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    results[name] = mae
    print(f"{name} MAE: {mae:.2f}")

# Compare performance
plt.figure(figsize=(8, 5))
plt.bar(results.keys(), results.values(), color=['skyblue', 'salmon', 'lightgreen'])
plt.ylabel("MAE (Lower is better)")
plt.title("Model Performance Comparison")
plt.show()

## Summary of Findings
- **Speed is Key:** Speed has the highest correlation with win percentage, suggesting that moving first is a major advantage in these simulated battles.
- **Legendary Status:** While Legendary Pokemon are strong, high Attack and Speed in non-legendaries still result in high win rates.
- **Model Performance:** Random Forest and Gradient Boosting significantly outperform Linear Regression, capturing the non-linear interactions between stats better.